# Approach 3 — Composite Archetype Scoring

## What this approach does

Rather than using hard thresholds (Approach 1) or letting an algorithm find clusters
(Approach 2), this notebook scores every practice against every archetype simultaneously.
Each practice receives an **affinity score from 0–100** for each of the four model bands
and a separate **size index** for each of the four size bands.

The archetype assigned is the one with the highest affinity score. But crucially, the
full score profile is preserved — so you can see *how strongly* a practice fits its
primary archetype and which secondary archetypes it has meaningful affinity with.

## Why use this approach?

| Strength | Detail |
|----------|--------|
| Richer than hard rules | Produces a continuous score, not just a binary in/out decision |
| Identifies boundary practices | Low confidence gap between top two scores = practice is genuinely between archetypes |
| Explainable | Every score is a weighted sum of named components — you can see exactly why a practice scored the way it did |
| Tunable | Weights can be adjusted to reflect business priorities without retraining a model |
| Directly creates archetypes | Unlike the previous modelling approach, this is designed to classify — not to predict income |

## How affinity scoring works

For each practice and each archetype, we compute a set of **signal components** —
normalised measures (0–100, based on percentile rank across the portfolio) that each
capture one aspect of the archetype's definition. These are then combined into a single
affinity score using weights that reflect how important each signal is to that archetype.

### Model archetype signal components

| Component | What it measures |
|-----------|------------------|
| `nhs_share_score` | Percentile rank of NHS share of income (100 = most NHS-heavy) |
| `private_share_score` | Inverse of above (100 = most private-heavy) |
| `private_intensity_score` | Percentile rank of private income per chair |
| `nhs_income_score` | Percentile rank of UDA volume |
| `balance_score` | Proximity to 50/50 NHS/private split (100 = perfectly balanced) |
| `specialist_score` | Referral patient volume (70%) + private income per chair (30%) — direct specialist/referral signal |

### Affinity weights per archetype

| Archetype | nhs_share | private_share | private_intensity | nhs_income | balance | specialist |
|-----------|-----------|---------------|-------------------|-----|---------|------------|
| NHS Led | **0.50** | 0.00 | 0.00 | **0.30** | 0.00 | 0.20 (inverted) |
| Balanced Mixed | 0.00 | 0.00 | 0.10 | 0.10 | **0.60** | 0.20 (inverted) |
| Private Led Mixed | 0.00 | **0.45** | **0.35** | 0.00 | 0.00 | 0.20 (inverted) |
| Specialist/Referral Hub | 0.00 | 0.25 | **0.35** | 0.00 | 0.00 | **0.40** |

Weights sum to 1.0 per archetype. They can be adjusted in the `WEIGHTS` dictionary below.

### Size index

A single continuous **size index** (0–100) is computed as a weighted average of
percentile ranks for surgeries (40%), staff headcount (40%), and contracted dentist
hours (20%). Practices are then assigned to size bands by quartile-cutting this index.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

BLUE   = '#4C72B0'
ORANGE = '#DD8452'
GREEN  = '#55A868'
RED    = '#C44E52'
GREY   = '#8C8C8C'
PURPLE = '#8172B2'

sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

SIZE_ORDER  = ['Small/Foundation', 'Medium/Core', 'Large/Advanced', 'Flagship']
MODEL_ORDER = ['NHS Led', 'Balanced Mixed', 'Private Led Mixed', 'Specialist/Referral Hub']
NA_ZONE_PAIRS = {('NHS Led', 'Large/Advanced'), ('NHS Led', 'Flagship')}

# Archetype affinity weights — adjust to reflect business priorities
#   Keys must match the component names computed below
WEIGHTS = {
    'NHS Led': {
        'nhs_share_score':      0.50,
        'nhs_income_score':            0.30,
        'anti_specialist_score': 0.20,
    },
    'Balanced Mixed': {
        'balance_score':         0.60,
        'anti_specialist_score': 0.20,
        'private_intensity_score': 0.10,
        'nhs_income_score':             0.10,
    },
    'Private Led Mixed': {
        'private_share_score':    0.45,
        'private_intensity_score': 0.35,
        'anti_specialist_score':  0.20,
    },
    'Specialist/Referral Hub': {
        'specialist_score':       0.40,
        'private_intensity_score': 0.35,
        'private_share_score':    0.25,
    },
}

df = pd.read_csv('master.csv')
print(f'Loaded {len(df):,} practices')

---
## Feature engineering

In [ ]:
df['nhs_income_est']   = df['nhsincome'].fillna(0)
df['private_income']   = df['privateincome'].fillna(0)
df['total_income_est'] = df['private_income'] + df['nhs_income_est']

df['nhs_share'] = np.where(
    df['total_income_est'] > 0,
    df['nhs_income_est'] / df['total_income_est'],
    np.nan
)
df['private_share'] = 1 - df['nhs_share'].fillna(0)

df['private_income_per_chair'] = df['private_income'] / df['numberofchairs'].replace(0, np.nan)
df['has_hygienist']            = (df['position_hygienist'] > 0).astype(float)

# Specialist composite: referral patient volume (primary) + private income per chair (secondary)
# Both are percentile-ranked to 0-1 so the weights are comparable.
pic_norm      = df['private_income_per_chair'].rank(pct=True).fillna(0)

df['total_referral_patients'] = (
    df['noofpatients_private_referral'].fillna(0) +
    df['noofpatients_nhs_referral'].fillna(0)
)
referral_norm = df['total_referral_patients'].rank(pct=True).fillna(0)

# 70% referral volume, 30% private income intensity
df['specialist_raw'] = (0.7 * referral_norm + 0.3 * pic_norm).clip(0, 1)

print('Feature engineering complete')
print(f'  NHS share range : {df["nhs_share"].min():.3f} - {df["nhs_share"].max():.3f}')
print(f'  Specialist raw  : {df["specialist_raw"].min():.2f} - {df["specialist_raw"].max():.2f}')

---
## Signal components

All components are expressed as **percentile ranks (0–100)** across the portfolio.
This ensures that a score of 50 always means "at the median", and the scores are
comparable across components with very different raw scales (e.g. NHS income vs income per chair).

The `balance_score` is different — it measures *proximity to the 50/50 NHS/private
midpoint* rather than a directional rank. A practice that is exactly 50% NHS / 50%
private gets a score of 100; practices at the extreme NHS or private end score near 0.

In [ ]:
def pct_rank(series):
    """Percentile rank scaled to 0-100, NaN -> 50 (neutral)."""
    return series.rank(pct=True).fillna(0.5) * 100

df['nhs_share_score']       = pct_rank(df['nhs_share'])          # high = more NHS
df['private_share_score']   = pct_rank(df['private_share'])       # high = more private
df['private_intensity_score'] = pct_rank(df['private_income_per_chair'])  # high = more private income per chair
df['nhs_income_score']             = pct_rank(df['nhsincome'])                 # high = more NHS income
df['specialist_score']      = pct_rank(df['specialist_raw'])      # high = more specialist signals
df['anti_specialist_score'] = 100 - df['specialist_score']        # high = less specialist (used by NHS Led / Balanced)

# Balance score: 100 = perfectly 50/50, 0 = entirely NHS or entirely private
df['balance_score'] = (1 - (df['nhs_share'].fillna(0.5) - 0.5).abs() / 0.5) * 100

component_cols = [
    'nhs_share_score', 'private_share_score', 'private_intensity_score',
    'nhs_income_score', 'balance_score', 'specialist_score', 'anti_specialist_score',
]

print('Component score summary (0-100):')
print(df[component_cols].describe().round(1).to_string())

# Visual: component distributions
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
fig.suptitle('Signal Component Distributions (0-100 Percentile Scale)', fontsize=13, fontweight='bold')
colors = sns.color_palette('Set2', len(component_cols))
for ax, col, color in zip(axes.flat, component_cols, colors):
    ax.hist(df[col], bins=20, color=color, edgecolor='white')
    ax.set_title(col.replace('_score','').replace('_',' '), fontsize=9)
    ax.set_xlabel('Score (0-100)')
axes.flat[-1].set_visible(False)
plt.tight_layout()
plt.show()

---
## Compute affinity scores

For each practice and each model archetype, the affinity score is the **weighted average
of its component scores**, where the weights are defined in the `WEIGHTS` dictionary at
the top of this notebook.

A score of **100** would mean this practice is at the top of the portfolio on every
signal component that defines this archetype. A score of **50** is the portfolio average.

In [ ]:
def compute_affinity(df, weights_dict):
    """Compute a weighted affinity score for each archetype. Returns DataFrame of scores."""
    scores = {}
    for archetype, weights in weights_dict.items():
        score = sum(df[component] * weight for component, weight in weights.items())
        scores[archetype] = score
    return pd.DataFrame(scores, index=df.index)

affinity = compute_affinity(df, WEIGHTS)

# Assign primary archetype
df['archetype_model']  = affinity.idxmax(axis=1)
df['affinity_primary'] = affinity.max(axis=1).round(1)

# Second-highest affinity (for confidence gap)
df['affinity_secondary'] = affinity.apply(lambda row: row.nlargest(2).iloc[1], axis=1).round(1)
df['affinity_confidence_gap'] = (df['affinity_primary'] - df['affinity_secondary']).round(1)

# Low confidence: practices where top two archetypes are close
LOW_CONFIDENCE_THRESHOLD = 10  # gap < 10 points = boundary practice
df['archetype_blend'] = df['affinity_confidence_gap'] < LOW_CONFIDENCE_THRESHOLD

# Attach individual scores for inspection
for arch in MODEL_ORDER:
    df[f'affinity_{arch.lower().replace("/","_").replace(" ","_")}'] = affinity[arch].round(1)

print('Affinity scores computed')
print(f'Blend practices (confidence gap < {LOW_CONFIDENCE_THRESHOLD}): '
      f'{df["archetype_blend"].sum()} ({df["archetype_blend"].mean()*100:.1f}%)')
print()
print('Model band distribution (before N/A enforcement):')
print(df['archetype_model'].value_counts().reindex(MODEL_ORDER).to_string())

---
## Size index and size band assignment

A single **size index** (0–100) combines three capacity signals with fixed weights.
Practices are then cut into four size bands at the portfolio quartiles of this index,
so the bands always represent equal shares of the portfolio regardless of how the
practice size distribution shifts with new data.

In [ ]:
SIZE_WEIGHTS = {
    'numberofsurgeries':       0.40,
    'unique_staff_ids':        0.40,
    'contractualhours_dentist': 0.20,
}

df['size_index'] = sum(
    pct_rank(df[col]) * weight
    for col, weight in SIZE_WEIGHTS.items()
)

# Cut at portfolio quartiles so bands are always ~25% each
q25, q50, q75 = df['size_index'].quantile([0.25, 0.50, 0.75])
df['archetype_size'] = pd.cut(
    df['size_index'],
    bins=[-np.inf, q25, q50, q75, np.inf],
    labels=SIZE_ORDER,
)

print(f'Size index quartiles:  P25={q25:.1f}  P50={q50:.1f}  P75={q75:.1f}')
print()
print('Size band distribution:')
print(df['archetype_size'].value_counts().reindex(SIZE_ORDER).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Size Index Distribution', fontweight='bold')

axes[0].hist(df['size_index'], bins=30, color=BLUE, edgecolor='white')
for q, label in [(q25, 'P25'), (q50, 'P50'), (q75, 'P75')]:
    axes[0].axvline(q, color=RED, lw=1.5, ls='--')
    axes[0].text(q + 0.3, axes[0].get_ylim()[1] * 0.9, label, color=RED, fontsize=8)
axes[0].set_xlabel('Size index (0-100)')
axes[0].set_title('Size index with quartile cut-offs')

counts = df['archetype_size'].value_counts().reindex(SIZE_ORDER)
colors = sns.color_palette('Set2', 4)
bars = axes[1].bar(counts.index, counts.values, color=colors, edgecolor='white')
axes[1].bar_label(bars, padding=3)
axes[1].set_title('Practice count per size band')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

---
## N/A zone enforcement

In [ ]:
for model_val, size_val in NA_ZONE_PAIRS:
    mask = (df['archetype_model'] == model_val) & (df['archetype_size'] == size_val)
    n = mask.sum()
    df.loc[mask, 'archetype_model'] = 'Balanced Mixed'
    if n:
        print(f'N/A zone: reclassified {n} [{model_val} x {size_val}] -> Balanced Mixed')

df['archetype_score'] = df['archetype_size'].astype(str) + ' | ' + df['archetype_model']

---
## 4 × 4 Matrix

In [ ]:
ct = pd.crosstab(
    df['archetype_size'], df['archetype_model']
).reindex(index=SIZE_ORDER, columns=MODEL_ORDER, fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Scoring-Based Archetype Matrix', fontsize=13, fontweight='bold')

# Count heatmap
sns.heatmap(ct, ax=axes[0], annot=True, fmt='d', cmap='Blues',
            linewidths=0.5, linecolor='white', cbar_kws={'shrink': 0.7})
axes[0].set_title('Practice count')
axes[0].set_xlabel('Model')
axes[0].set_ylabel('Size')
axes[0].tick_params(axis='x', rotation=20)
axes[0].tick_params(axis='y', rotation=0)
for model_val, size_val in NA_ZONE_PAIRS:
    ri, ci = SIZE_ORDER.index(size_val), MODEL_ORDER.index(model_val)
    axes[0].add_patch(plt.Rectangle((ci, ri), 1, 1, fill=True, color='#d0d0d0', zorder=3, alpha=0.8))
    axes[0].text(ci+0.5, ri+0.5, 'N/A', ha='center', va='center', fontsize=9, color='grey', zorder=4)

# Average confidence gap heatmap
gap_pivot = (
    df.groupby(['archetype_size', 'archetype_model'], observed=True)['affinity_confidence_gap']
    .mean()
    .unstack(fill_value=np.nan)
    .reindex(index=SIZE_ORDER, columns=MODEL_ORDER)
)
annot = gap_pivot.applymap(lambda v: f'{v:.0f}' if not np.isnan(v) else '')
sns.heatmap(gap_pivot, ax=axes[1], annot=annot, fmt='', cmap='YlGn',
            linewidths=0.5, linecolor='white', cbar_kws={'shrink': 0.7})
axes[1].set_title('Avg confidence gap (higher = clearer archetype fit)')
axes[1].set_xlabel('Model')
axes[1].set_ylabel('Size')
axes[1].tick_params(axis='x', rotation=20)
axes[1].tick_params(axis='y', rotation=0)
for model_val, size_val in NA_ZONE_PAIRS:
    ri, ci = SIZE_ORDER.index(size_val), MODEL_ORDER.index(model_val)
    axes[1].add_patch(plt.Rectangle((ci, ri), 1, 1, fill=True, color='#d0d0d0', zorder=3, alpha=0.8))
    axes[1].text(ci+0.5, ri+0.5, 'N/A', ha='center', va='center', fontsize=9, color='grey', zorder=4)

plt.tight_layout()
plt.show()

print(ct.to_string())

---
## Affinity score profiles

The radar-style parallel coordinates chart below shows the **mean affinity score for
each archetype, broken down by component**. A well-designed scoring system should show
clearly differentiated profiles — each archetype should score highest on the components
that define it.

If two archetypes have similar profiles here, it means the component weights need
adjusting to better discriminate between them.

In [ ]:
# Mean component scores per assigned archetype
archetype_profiles = (
    df.groupby('archetype_model')[component_cols]
    .mean()
    .reindex(MODEL_ORDER)
    .round(1)
)

display(archetype_profiles)

# Profile chart
fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(component_cols))
palette = sns.color_palette('Set2', 4)

for arch, color in zip(MODEL_ORDER, palette):
    if arch in archetype_profiles.index:
        ax.plot(x, archetype_profiles.loc[arch, component_cols],
                marker='o', lw=2, color=color, label=arch)

ax.set_xticks(x)
ax.set_xticklabels([c.replace('_score','').replace('_',' ') for c in component_cols],
                   rotation=15, ha='right')
ax.set_ylabel('Mean component score (0-100)')
ax.set_title('Archetype Signal Profiles — Mean Component Scores', fontweight='bold')
ax.axhline(50, color='grey', lw=0.8, ls='--', label='Portfolio median (50)')
ax.legend(fontsize=9, loc='lower right')
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

---
## Confidence and blend analysis

**Confidence gap** = primary affinity score minus secondary affinity score.
A high gap means the practice is a clear fit for its archetype. A gap below 10 points
means the practice is genuinely sitting between two archetypes — these *blend* practices
deserve closer inspection before acting on their label.

This is one of the key advantages of scoring over hard rules: the rules approach will
assign a practice to a band regardless of how close it is to the boundary. The scoring
approach surfaces that ambiguity explicitly.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
fig.suptitle('Confidence Gap Analysis', fontsize=13, fontweight='bold')

# Gap distribution
axes[0].hist(df['affinity_confidence_gap'], bins=25, color=BLUE, edgecolor='white')
axes[0].axvline(LOW_CONFIDENCE_THRESHOLD, color=RED, lw=1.8, ls='--',
                label=f'Blend threshold ({LOW_CONFIDENCE_THRESHOLD})')
axes[0].set_xlabel('Confidence gap (points)')
axes[0].set_title('Distribution of confidence gaps')
axes[0].legend(fontsize=8)

# Blend % per model archetype
blend_rate = (
    df.groupby('archetype_model')['archetype_blend']
    .mean() * 100
).reindex(MODEL_ORDER)
bars = axes[1].bar(blend_rate.index, blend_rate.values, color=ORANGE, edgecolor='white')
axes[1].bar_label(bars, fmt='%.1f%%', padding=3)
axes[1].set_title('% blend practices per model band')
axes[1].set_ylabel('%')
axes[1].tick_params(axis='x', rotation=20)

# Primary vs secondary affinity scatter
colors_scatter = [sns.color_palette('Set2', 4)[MODEL_ORDER.index(m)] if m in MODEL_ORDER else GREY
                  for m in df['archetype_model']]
axes[2].scatter(df['affinity_secondary'], df['affinity_primary'],
                c=colors_scatter, alpha=0.4, s=25)
axes[2].plot([0, 100], [0, 100], color='grey', lw=0.8, ls='--')
axes[2].axhline(df['affinity_primary'].mean(), color=RED, lw=1, ls=':')
axes[2].set_xlabel('Secondary affinity score')
axes[2].set_ylabel('Primary affinity score')
axes[2].set_title('Primary vs secondary affinity')

plt.tight_layout()
plt.show()

n_blend = df['archetype_blend'].sum()
print(f'Total blend practices: {n_blend} ({n_blend/len(df)*100:.1f}%)')
print(f'Mean confidence gap:   {df["affinity_confidence_gap"].mean():.1f} points')

---
## Full affinity score view

The table below shows, for each practice, all four affinity scores side by side.
The primary archetype column is highlighted. This view is useful for understanding
how a practice differs from others in the same archetype cell.

In [ ]:
affinity_score_cols = [
    'affinity_nhs_led',
    'affinity_balanced_mixed',
    'affinity_private_led_mixed',
    'affinity_specialist_referral_hub',
]

# Check which columns exist (name sanitisation may vary)
affinity_score_cols = [c for c in df.columns if c.startswith('affinity_') and
                       c not in ('affinity_primary', 'affinity_secondary', 'affinity_confidence_gap')]

display_cols = (
    ['practicename', 'region', 'archetype_size', 'archetype_model',
     'affinity_primary', 'affinity_secondary', 'affinity_confidence_gap', 'archetype_blend']
    + affinity_score_cols
)

sample = df[display_cols].sort_values('archetype_confidence_gap'
    if 'archetype_confidence_gap' in df.columns else 'affinity_confidence_gap')

# Show the 10 most ambiguous blend practices first, then 10 most confident
blend_sample = sample[sample['archetype_blend']].head(10)
clear_sample = sample[~sample['archetype_blend']].tail(10)

if len(blend_sample):
    print(f'10 most ambiguous blend practices (lowest confidence gap):')
    display(
        blend_sample.reset_index(drop=True)
        .style.format({c: '{:.1f}' for c in affinity_score_cols +
                       ['affinity_primary','affinity_secondary','affinity_confidence_gap']})
        .background_gradient(subset=affinity_score_cols, cmap='YlOrRd', axis=1)
    )

print(f'\n10 most clearly classified practices (highest confidence gap):')
display(
    clear_sample.reset_index(drop=True)
    .style.format({c: '{:.1f}' for c in affinity_score_cols +
                   ['affinity_primary','affinity_secondary','affinity_confidence_gap']})
    .background_gradient(subset=affinity_score_cols, cmap='YlOrRd', axis=1)
)

---
## Agreement with rules-based labels

Comparing the scoring output to the rules-based labels shows where the two approaches
agree. Disagreements are most likely at boundary practices — the scoring approach will
flag these as blend cases (low confidence gap), while the rules approach silently
assigns them to one side of the threshold.

> Run `01_rules_based.ipynb` first if `archetypes_rules.csv` does not exist.

In [ ]:
import os

if os.path.exists('archetypes_rules.csv'):
    rules = pd.read_csv('archetypes_rules.csv')[['practicekey', 'archetype_size', 'archetype_model']]
    rules.columns = ['practicekey', 'rules_size', 'rules_model']
    merged = df.merge(rules, on='practicekey', how='left')

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle('Scoring vs Rules-Based Agreement', fontsize=13, fontweight='bold')

    size_agree  = pd.crosstab(
        merged['rules_size'], merged['archetype_size'].astype(str)
    ).reindex(index=SIZE_ORDER, columns=SIZE_ORDER, fill_value=0)
    model_agree = pd.crosstab(
        merged['rules_model'], merged['archetype_model']
    ).reindex(index=MODEL_ORDER, columns=MODEL_ORDER, fill_value=0)

    sns.heatmap(size_agree,  ax=axes[0], annot=True, fmt='d', cmap='Greens',
                linewidths=0.5, linecolor='white')
    axes[0].set_title('Size agreement')
    axes[0].set_xlabel('Scoring')
    axes[0].set_ylabel('Rules-Based')
    axes[0].tick_params(axis='x', rotation=20)
    axes[0].tick_params(axis='y', rotation=0)

    sns.heatmap(model_agree, ax=axes[1], annot=True, fmt='d', cmap='Oranges',
                linewidths=0.5, linecolor='white')
    axes[1].set_title('Model agreement')
    axes[1].set_xlabel('Scoring')
    axes[1].set_ylabel('Rules-Based')
    axes[1].tick_params(axis='x', rotation=20)
    axes[1].tick_params(axis='y', rotation=0)

    plt.tight_layout()
    plt.show()

    size_pct  = (merged['archetype_size'].astype(str) == merged['rules_size']).mean()  * 100
    model_pct = (merged['archetype_model'] == merged['rules_model']).mean() * 100
    print(f'Size  agreement: {size_pct:.1f}%')
    print(f'Model agreement: {model_pct:.1f}%')

    # Among blend practices, what % disagree with rules?
    blend_mask  = merged['archetype_blend']
    blend_disagree = (merged.loc[blend_mask, 'archetype_model'] != merged.loc[blend_mask, 'rules_model']).mean() * 100
    print(f'\nAmong blend practices, {blend_disagree:.1f}% disagree with rules-based labels')
    print('(Higher = scoring is surfacing genuine boundary cases that rules assigned arbitrarily)')
else:
    print('archetypes_rules.csv not found. Run 01_rules_based.ipynb first.')

---
## Save output

In [ ]:
out_cols = (
    ['practicekey', 'practicename', 'region',
     'numberofsurgeries', 'unique_staff_ids',
     'private_income', 'nhs_income_est', 'total_income_est', 'nhs_share',
     'size_index', 'archetype_size', 'archetype_model', 'archetype_score',
     'affinity_primary', 'affinity_secondary', 'affinity_confidence_gap', 'archetype_blend']
    + affinity_score_cols
)
df[out_cols].to_csv('archetypes_scoring.csv', index=False)
print(f'Saved archetypes_scoring.csv  ({len(df):,} rows, {len(out_cols)} columns)')
df[out_cols].head()